# Week 1: NumPy Basics for Computer Vision

**Computer Vision Fundamentals** — Temasek Poly IIT AAI

*(Teacher copy — full solutions and explanations.)*

Images in OpenCV are **NumPy arrays**. This notebook covers NumPy operations you will use again and again in CV: reshaping, stacking, conditional logic (`np.where`), clipping, and masking.

---
## 1.1 Setup and why shape matters

In computer vision, an image is a 2D (grayscale) or 3D (colour) NumPy array with **shape** `(height, width)` or `(height, width, channels)`. Knowing and changing shape is essential: you might **flatten** a patch to feed a classifier, **reshape** a list of pixels back into an image, or add a **batch dimension** for deep learning (e.g. from `(H, W, C)` to `(1, H, W, C)`).

- **`arr.shape`** gives the dimensions as a tuple. Use it to get height, width, or to check that two images match before concatenating.
- **`arr.reshape(new_shape)`** returns a new view (when possible) with the given dimensions. The total number of elements must stay the same. Using **`-1`** for one dimension lets NumPy infer it from the rest (e.g. `reshape(-1)` flattens to 1D; `reshape(3, -1)` keeps 3 rows and infers the number of columns).

In [ ]:
import numpy as np

# Small 2D array (like a tiny grayscale image)
gray = np.array([[10, 20, 30], [40, 50, 60]], dtype=np.uint8)
print('Shape:', gray.shape)
print('Flatten to 1D:', gray.reshape(-1))
print('Reshape to (3, 2):', gray.reshape(3, 2))

---
## 1.2 Indexing and slicing — ROIs and regions

**Why this matters in CV:** You often need to work on a **region of interest (ROI)**—a rectangular patch of the image (e.g. a detected face, a license plate, or a crop for further processing). NumPy indexing and slicing is how you select that region.

**Convention:** For a 2D array (grayscale image), the first index is the **row** (vertical, often called **y**), the second is the **column** (horizontal, **x**). So `img[y, x]` is the pixel at row `y`, column `x`. For a colour image (3D), `img[y, x, channel]` with channel 0, 1, 2 for BGR.

- **Single element:** `arr[row, col]` (or `arr[y, x]`). For 3D: `arr[y, x, channel]`.
- **Full row or column:** `arr[row, :]` gives that row; `arr[:, col]` gives that column.
- **Slicing a rectangle (ROI):** `arr[r1:r2, c1:c2]` gives rows from `r1` up to (but not including) `r2`, and columns from `c1` to `c2-1`. So a 100×100 patch starting at (50, 80) is `img[50:150, 80:180]`.
- **Negative indices:** `-1` means "last"; e.g. `img[-1, :]` is the last row, `img[:, -100:]` is the last 100 columns.
- **Step:** `img[::2, ::2]` takes every 2nd row and column (useful for fast downsampling).

**Important:** Slicing returns a **view** into the original array by default—if you modify the slice, you modify the original. Use `roi = img[y1:y2, x1:x2].copy()` when you need an independent copy of the ROI.

In [ ]:
# 2D array like a small grayscale image (4 rows, 5 cols)
img = np.array([[10, 11, 12, 13, 14],
                [20, 21, 22, 23, 24],
                [30, 31, 32, 33, 34],
                [40, 41, 42, 43, 44]], dtype=np.uint8)

print('Single pixel img[1, 2]:', img[1, 2])
print('Row 1 (img[1, :]):', img[1, :])
print('Column 2 (img[:, 2]):', img[:, 2])

# ROI: rows 1–2, columns 2–4 (inclusive of start, exclusive of end)
roi = img[1:3, 2:5]
print('ROI img[1:3, 2:5]:')
print(roi)

# Last row and last two columns
print('Last row:', img[-1, :])
print('Last two columns:')
print(img[:, -2:])

---
## 1.3 np.concatenate — join arrays along an axis

**Application:** Stitch two or more images together, or build a batch of images. For example, place two images of the same height **side by side** (concatenate along columns, `axis=1`) or **one above the other** (along rows, `axis=0`). In deep learning, you often stack many images along a new batch axis.

**Syntax:** `np.concatenate((a, b, ...), axis=...)`. The arrays must have the **same shape on every axis except the one you are concatenating along**. For 2D: `axis=0` = vertical (more rows), `axis=1` = horizontal (more columns). For a 3D image `(H, W, C)`, `axis=0` or `axis=1` still stack in the spatial dimensions; `axis=2` would stack along channels (e.g. adding an extra channel). If shapes don't match, NumPy raises an error.

In [ ]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print('Vertical (axis=0):')
print(np.concatenate((a, b), axis=0))
print('Horizontal (axis=1):')
print(np.concatenate((a, b), axis=1))

---
## 1.4 np.vstack and np.hstack — stack images into grids

**Application:** These are convenient shortcuts for stacking arrays **vertically** (`np.vstack`) or **horizontally** (`np.hstack`). In CV you use them to build a single image from several smaller ones—e.g. show original, filtered, and masked versions in one row, or stack multiple detected patches into a grid for visualization or debugging.

**Syntax:** `np.vstack((a, b, ...))` is equivalent to `np.concatenate((a, b, ...), axis=0)`; `np.hstack((a, b, ...))` is `axis=1`. You pass a **sequence** (tuple or list) of arrays. All arrays must have the same shape along the non-stacking dimension (for `vstack`, same number of columns; for `hstack`, same number of rows).

In [ ]:
row1 = np.array([[1, 1, 1], [1, 1, 1]])
row2 = np.array([[2, 2, 2], [2, 2, 2]])

grid_v = np.vstack((row1, row2))
grid_h = np.hstack((row1, row2))
print('vstack (vertical):')
print(grid_v)
print('hstack (horizontal):')
print(grid_h)

---
## 1.5 np.where — conditional values (thresholds, masks)

**Application:** In CV you often need to **threshold** pixels (e.g. bright = 255, dark = 0 for a binary mask) or **replace values** only where a condition holds. `np.where(condition, value_if_true, value_if_false)` returns an array of the same shape as `condition`: wherever the condition is True it takes `value_if_true`, otherwise `value_if_false`. The condition is typically a comparison on an array (e.g. `img > 128`), which gives a boolean array. You can use this for simple binarization, for building masks to use elsewhere, or for combining two images based on a mask.

In [ ]:
x = np.array([10, 100, 200, 50, 255])
binary = np.where(x > 128, 255, 0)
print('Original:', x)
print('Where x > 128 → 255 else 0:', binary)

# 2D example (like a tiny image)
img = np.array([[100, 200], [50, 250]])
mask = np.where(img > 128, 1, 0)
print('Mask (1 where > 128):')
print(mask)

---
## 1.6 np.clip — keep values in range

**Application:** When you do arithmetic on pixel values (e.g. add 50 for brightness, or multiply by 1.5), results can go **below 0** or **above 255**. For 8-bit images we must keep values in the range **[0, 255]**. If you don't clip, casting to `uint8` causes **overflow**: values above 255 wrap around (e.g. 260 becomes 4), which can create visible artifacts.

**Syntax:** `np.clip(arr, min_val, max_val)` returns an array (same shape as `arr`) with every element forced into `[min_val, max_val]`. Values below the minimum become the minimum; values above the maximum become the maximum. The typical pipeline after brightness/contrast is: work in float → `np.clip(arr, 0, 255)` → `.astype(np.uint8)`.

In [ ]:
vals = np.array([-10, 50, 200, 300])
clipped = np.clip(vals, 0, 255)
print('Original:', vals)
print('Clipped to [0, 255]:', clipped)

---
## 1.7 Boolean indexing and masks

**Application:** A **boolean mask** is an array of the same shape as your image with `True`/`False` values. You can use it to **select** only the pixels where the mask is True, or to **assign** new values only to those pixels. For example: get all pixel values in a segmented region, set background to black, or apply a filter only inside a mask.

**Syntax:** `arr[mask]` returns a **1D array** of all elements where `mask` is True (in row-major order). Assigning with `arr[mask] = value` (or `arr[mask] = other_values`) changes only those positions. The mask must be boolean and the same shape as `arr`. This is different from slicing: the result of `arr[mask]` is a flattened selection, not a 2D subregion.

In [ ]:
arr = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
mask = arr % 2 == 0  # True where even
print('Array:')
print(arr)
print('Values where mask is True:', arr[mask])

# Set all even positions to 0
arr_mod = arr.copy()
arr_mod[mask] = 0
print('After setting even to 0:')
print(arr_mod)

---
## 1.8 np.zeros_like, np.ones_like — same shape as another array

**Application:** You often need a new array that **matches the shape (and dtype)** of an existing one—e.g. a mask of zeros to fill in, an accumulator, or an output buffer. `np.zeros_like(arr)` and `np.ones_like(arr)` create arrays of zeros or ones with the same shape and dtype as `arr`. This avoids writing `np.zeros(img.shape, dtype=img.dtype)` and keeps your code correct if the image size or type changes.

In [ ]:
ref = np.array([[1, 2], [3, 4]])
zeros = np.zeros_like(ref)
ones = np.ones_like(ref)
print('Reference shape/dtype:', ref.shape, ref.dtype)
print('zeros_like:')
print(zeros)
print('ones_like:')
print(ones)

---
## 1.9 Summary

These operations form the core of array manipulation for CV:

- **shape / reshape**: Inspect and change dimensions; `-1` infers one dimension.
- **Indexing and slicing**: `img[y, x]`, `img[y1:y2, x1:x2]` for ROI; rows = first index, cols = second. Use `.copy()` when you need an independent ROI.
- **np.concatenate**: Join arrays along a given axis (0=vertical, 1=horizontal).
- **np.vstack / np.hstack**: Stack arrays vertically or horizontally (convenience for image grids).
- **np.where**: Conditional values (thresholds, binarization, masks).
- **np.clip**: Keep values in [min, max] (e.g. 0–255 for pixels).
- **Boolean indexing**: Select or set elements where a condition is true.
- **np.zeros_like / np.ones_like**: Create same-shape arrays for masks or buffers.